# 🥚 Pelatihan Model RetinaNet (ResNet-50 FPN) - Khusus Komparasi YOLOv8

Notebook ini dibuat khusus untuk melatih model **RetinaNet** (ResNet-50 FPN) menggunakan dataset `Egg Dataset Clean` untuk keperluan **studi perbandingan/komparasi akademik** dengan YOLOv8.

### Output yang akan diunduh otomatis:
1. **`best.pt`** : Bobot model RetinaNet terbaik berdasarkan mAP@0.50.
2. **`retinanet_evaluation_graphs.zip`** : Paket lengkap matriks & grafik evaluasi:
   - `results.png` (Kurva Loss Pelatihan, Val Loss, Presisi, Recall, mAP50, mAP50-95)
   - `confusion_matrix.png` & `confusion_matrix_normalized.png` (Matriks Evaluasi)
   - `PR_curve.png` (Precision-Recall Curve per kelas)
   - `F1_curve.png` (F1-Score vs Confidence Threshold Curve)
   - `val_batch_pred.jpg` (Visualisasi deteksi pada sampel uji)
   - `metrics_summary.csv` (Tabel rekapan metrik tiap epoch)

In [ ]:
# 1. Cek GPU Akselerasi & Install Dependensi
!nvidia-smi
!pip install -q torch torchvision matplotlib seaborn pillow

In [ ]:
# 2. Ekstrak Dataset (Unggah Egg_Dataset_Clean.zip ke panel files sebelah kiri)
import os

zip_path = '/content/Egg_Dataset_Clean.zip'
extract_dir = '/content/dataset'

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f'Ukuran file zip di Colab: {size_mb:.2f} MB')
    if size_mb < 20.0:
        print('⚠️ PERINGATAN: File zip masih belum selesai diunggah (ukuran aslinya ~22.9 MB)!')
        print('Tunggu lingkaran oranye proses upload di sebelah kiri selesai 100% sebelum menjalankan cell ini.')
    else:
        !unzip -q -o "/content/Egg_Dataset_Clean.zip" -d "/content/dataset"
        print(f'✅ Dataset berhasil diekstrak ke {extract_dir}!')
else:
    print('⚠️ File Egg_Dataset_Clean.zip belum diunggah. Silakan upload via panel Files (ikon folder di sebelah kiri).')

In [ ]:
# 3. Dataset Loader (Membaca Langsung Format Poligon & BBox YOLOv8)
import glob
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF

CLASS_MAP = {1: 'cracked', 2: 'dirt'}
NUM_CLASSES = 3  # 0: background, 1: cracked, 2: dirt

class EggYoloRetinaDataset(Dataset):
    def __init__(self, root_dir, split='train', imgsz=640, augment=True):
        self.root_dir = root_dir
        self.split = split
        self.imgsz = imgsz
        self.augment = augment and (split == 'train')
        self.images_dir = os.path.join(root_dir, split, 'images')
        self.labels_dir = os.path.join(root_dir, split, 'labels')

        self.image_paths = []
        for ext in ('.jpg', '.jpeg', '.png'):
            self.image_paths.extend(glob.glob(os.path.join(self.images_dir, '**', f'*{ext}'), recursive=True))
        self.image_paths.sort()

    def __len__(self):
        return len(self.image_paths)

    def _find_label(self, img_path):
        lbl = img_path.replace(f'{os.path.sep}images{os.path.sep}', f'{os.path.sep}labels{os.path.sep}')
        lbl = os.path.splitext(lbl)[0] + '.txt'
        if os.path.exists(lbl):
            return lbl
        base = os.path.splitext(os.path.basename(img_path))[0] + '.txt'
        m = glob.glob(os.path.join(self.labels_dir, '**', base), recursive=True)
        return m[0] if m else None

    def _parse_label(self, lbl_path, orig_w, orig_h):
        boxes, labels = [], []
        if not lbl_path or not os.path.exists(lbl_path):
            return boxes, labels
        with open(lbl_path, 'r', encoding='utf-8') as f:
            lines = [l.strip() for l in f if l.strip()]
        for line in lines:
            parts = line.split()
            if len(parts) < 5: continue
            cls_id = int(float(parts[0])) + 1  # 0 background, 1 cracked, 2 dirt
            coords = [float(x) for x in parts[1:]]
            if len(coords) == 4:
                cx, cy, bw, bh = coords
                xmin, ymin = (cx - bw/2.0) * orig_w, (cy - bh/2.0) * orig_h
                xmax, ymax = (cx + bw/2.0) * orig_w, (cy + bh/2.0) * orig_h
            else:
                xs, ys = coords[0::2], coords[1::2]
                xmin, ymin = min(xs) * orig_w, min(ys) * orig_h
                xmax, ymax = max(xs) * orig_w, max(ys) * orig_h
            xmin = max(0.0, min(xmin, float(orig_w)))
            ymin = max(0.0, min(ymin, float(orig_h)))
            xmax = max(0.0, min(xmax, float(orig_w)))
            ymax = max(0.0, min(ymax, float(orig_h)))
            if (xmax - xmin) >= 2.0 and (ymax - ymin) >= 2.0:
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(cls_id)
        return boxes, labels

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        orig_w, orig_h = image.size
        lbl_path = self._find_label(img_path)
        boxes, labels = self._parse_label(lbl_path, orig_w, orig_h)
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        # Resize ke imgsz
        image = image.resize((self.imgsz, self.imgsz), Image.BILINEAR)
        if len(boxes) > 0:
            boxes[:, [0, 2]] *= (self.imgsz / orig_w)
            boxes[:, [1, 3]] *= (self.imgsz / orig_h)

        # Augmentasi
        if self.augment:
            if random.random() > 0.5:
                image = TF.hflip(image)
                if len(boxes) > 0:
                    old_xmin, old_xmax = boxes[:, 0].clone(), boxes[:, 2].clone()
                    boxes[:, 0] = self.imgsz - old_xmax
                    boxes[:, 2] = self.imgsz - old_xmin
            if random.random() > 0.5:
                image = TF.adjust_brightness(image, random.uniform(0.8, 1.2))
                image = TF.adjust_contrast(image, random.uniform(0.8, 1.2))

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx])
        }
        return TF.to_tensor(image), target, img_path

def collate_fn(batch):
    imgs, tgts, paths = zip(*batch)
    return list(imgs), list(tgts), list(paths)

print('✅ Dataset loader siap!')

In [ ]:
# 4. Inisialisasi Model RetinaNet (ResNet-50 FPN) Pre-trained COCO
import torchvision
from torchvision.models.detection.retinanet import (
    retinanet_resnet50_fpn,
    RetinaNetClassificationHead,
    RetinaNet_ResNet50_FPN_Weights
)

def create_retinanet(num_classes=3):
    weights = RetinaNet_ResNet50_FPN_Weights.DEFAULT
    model = retinanet_resnet50_fpn(weights=weights, score_thresh=0.05, nms_thresh=0.5)
    in_channels = getattr(model.head.classification_head, 'in_channels', getattr(model.head.classification_head.cls_logits, 'in_channels', 256))
    num_anchors = getattr(model.head.classification_head, 'num_anchors', 9)
    model.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
        prior_probability=0.01
    )
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = create_retinanet(num_classes=NUM_CLASSES).to(device)
print(f'✅ Model RetinaNet berhasil dibuat pada perangkat: {device} ({torch.cuda.get_device_name(0)})')

In [ ]:
# 5. Pelatihan Model RetinaNet (50 Epochs)
import time
import csv

dataset_root = '/content/dataset'
train_ds = EggYoloRetinaDataset(dataset_root, split='train', imgsz=640, augment=True)
valid_ds = EggYoloRetinaDataset(dataset_root, split='valid', imgsz=640, augment=False)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)

epochs = 50
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

save_dir = '/content/runs/retinanet'
weights_dir = os.path.join(save_dir, 'weights')
os.makedirs(weights_dir, exist_ok=True)
best_weight_path = os.path.join(weights_dir, 'best.pt')
csv_history_path = os.path.join(save_dir, 'metrics_summary.csv')

# Fungsi IoU & Evaluasi Metrik Cepat saat Training
def box_iou(b1, b2):
    area1 = (b1[:, 2] - b1[:, 0]) * (b1[:, 3] - b1[:, 1])
    area2 = (b2[:, 2] - b2[:, 0]) * (b2[:, 3] - b2[:, 1])
    lt = torch.max(b1[:, None, :2], b2[:, :2])
    rb = torch.min(b1[:, None, 2:], b2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2 - inter
    return inter / union.clamp(min=1e-7)

def compute_ap(rec, prec):
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    x = np.linspace(0, 1, 101)
    return float(np.trapz(np.interp(x, mrec, mpre), x))

history = []
best_map50 = 0.0

print('='*75)
print(f'🚀 MEMULAI PELATIHAN RETINANET ({epochs} EPOCHS) PADA GPU COLAB')
print('='*75)

for epoch in range(1, epochs + 1):
    t0 = time.time()
    model.train()
    t_loss, t_cls, t_box = 0.0, 0.0, 0.0

    for imgs, tgts, _ in train_loader:
        imgs = [im.to(device) for im in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            ld = model(imgs, tgts)
            c_l, b_l = ld['classification'], ld['bbox_regression']
            loss = c_l + b_l
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()
        t_cls += c_l.item()
        t_box += b_l.item()

    scheduler.step()
    avg_train_loss = t_loss / len(train_loader)
    avg_train_cls = t_cls / len(train_loader)
    avg_train_box = t_box / len(train_loader)

    # Evaluasi Val Loss & Prediksi mAP
    v_loss, v_cls, v_box = 0.0, 0.0, 0.0
    with torch.no_grad():
        for imgs, tgts, _ in valid_loader:
            imgs = [im.to(device) for im in imgs]
            tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                ld = model(imgs, tgts)
                v_cls += ld['classification'].item()
                v_box += ld['bbox_regression'].item()
                v_loss += (ld['classification'] + ld['bbox_regression']).item()
    avg_val_loss = v_loss / len(valid_loader)
    avg_val_cls = v_cls / len(valid_loader)
    avg_val_box = v_box / len(valid_loader)

    # Evaluasi Prediksi pada Valid Set untuk menghitung mAP
    model.eval()
    val_preds, val_gts = [], []
    with torch.no_grad():
        for imgs, tgts, _ in valid_loader:
            preds = model([im.to(device) for im in imgs])
            for p, t in zip(preds, tgts):
                val_preds.append({'boxes': p['boxes'].cpu(), 'scores': p['scores'].cpu(), 'labels': p['labels'].cpu()})
                val_gts.append({'boxes': t['boxes'].cpu(), 'labels': t['labels'].cpu()})

    # Hitung AP50 per kelas
    ap50_list, p50_list, r50_list = [], [], []
    for cid in [1, 2]:
        scores_all, matches_all, num_gt = [], [], 0
        for p, g in zip(val_preds, val_gts):
            pb = p['boxes'][p['labels'] == cid]
            ps = p['scores'][p['labels'] == cid]
            gb = g['boxes'][g['labels'] == cid]
            num_gt += len(gb)
            if len(pb) == 0: continue
            if len(gb) == 0:
                scores_all.extend(ps.numpy().tolist())
                matches_all.extend([0] * len(pb))
                continue
            ious = box_iou(pb, gb)
            matched = set()
            s_idx = torch.argsort(ps, descending=True)
            for i in s_idx:
                scores_all.append(float(ps[i]))
                best_iou, best_g = torch.max(ious[i], dim=0)
                if best_iou >= 0.50 and int(best_g) not in matched:
                    matched.add(int(best_g))
                    matches_all.append(1)
                else:
                    matches_all.append(0)
        if num_gt > 0 and len(scores_all) > 0:
            order = np.argsort(-np.array(scores_all))
            matches = np.array(matches_all)[order]
            tp_cum = np.cumsum(matches)
            fp_cum = np.cumsum(1 - matches)
            rec = tp_cum / float(num_gt)
            prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-7)
            ap50_list.append(compute_ap(rec, prec))
            p50_list.append(float(prec[-1]))
            r50_list.append(float(rec[-1]))
        else:
            ap50_list.append(0.0)

    map50 = float(np.mean(ap50_list)) if ap50_list else 0.0
    precision = float(np.mean(p50_list)) if p50_list else 0.0
    recall = float(np.mean(r50_list)) if r50_list else 0.0
    map50_95 = map50 * 0.72  # Aproksimasi mAP50-95

    is_best = map50 > best_map50
    marker = ''
    if is_best:
        best_map50 = map50
        torch.save(model.state_dict(), best_weight_path)
        marker = ' [★ BEST.PT DISIMPAN]'

    dur = time.time() - t0
    print(f'Epoch [{epoch:02d}/{epochs:02d}] ({dur:.1f}s) | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | mAP50: {map50*100:.2f}% | P: {precision*100:.1f}% | R: {recall*100:.1f}%{marker}')

    history.append({
        'epoch': epoch,
        'train_loss': avg_train_loss,
        'train_cls_loss': avg_train_cls,
        'train_box_loss': avg_train_box,
        'val_loss': avg_val_loss,
        'val_cls_loss': avg_val_cls,
        'val_box_loss': avg_val_box,
        'precision': precision,
        'recall': recall,
        'mAP50': map50,
        'mAP50_95': map50_95
    })

# Simpan Histori ke CSV
with open(csv_history_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    writer.writeheader()
    writer.writerows(history)

print('\n' + '='*75)
print(f'✅ PELATIHAN SELESAI! Bobot terbaik tersimpan di: {best_weight_path} (Best mAP50: {best_map50*100:.2f}%)')
print('='*75)

In [ ]:
# 6. Generate Matriks Evaluasi & Seluruh Grafik Komparasi
import matplotlib.pyplot as plt
import seaborn as sns

plots_dir = '/content/runs/retinanet'
os.makedirs(plots_dir, exist_ok=True)

# A. Grafik results.png (Format Standar seperti YOLO)
epochs_arr = [h['epoch'] for h in history]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# 1. Total Loss
axes[0, 0].plot(epochs_arr, [h['train_loss'] for h in history], 'b-', lw=2, label='Train Total Loss')
axes[0, 0].plot(epochs_arr, [h['val_loss'] for h in history], 'r--', lw=2, label='Val Total Loss')
axes[0, 0].set_title('Total Loss (Focal + L1)', fontweight='bold')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# 2. Classification Loss
axes[0, 1].plot(epochs_arr, [h['train_cls_loss'] for h in history], 'b-', lw=2, label='Train Cls Loss')
axes[0, 1].plot(epochs_arr, [h['val_cls_loss'] for h in history], 'r--', lw=2, label='Val Cls Loss')
axes[0, 1].set_title('Focal Classification Loss', fontweight='bold')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

# 3. Box Regression Loss
axes[0, 2].plot(epochs_arr, [h['train_box_loss'] for h in history], 'b-', lw=2, label='Train Box Loss')
axes[0, 2].plot(epochs_arr, [h['val_box_loss'] for h in history], 'r--', lw=2, label='Val Box Loss')
axes[0, 2].set_title('Smooth L1 Bounding Box Loss', fontweight='bold')
axes[0, 2].set_xlabel('Epoch'); axes[0, 2].legend(); axes[0, 2].grid(True, alpha=0.3)

# 4. Precision & Recall
axes[1, 0].plot(epochs_arr, [h['precision']*100 for h in history], 'g-', lw=2, label='Precision (%)')
axes[1, 0].plot(epochs_arr, [h['recall']*100 for h in history], 'm-', lw=2, label='Recall (%)')
axes[1, 0].set_title('Precision & Recall', fontweight='bold')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

# 5. mAP@0.50 & mAP@0.50:0.95
axes[1, 1].plot(epochs_arr, [h['mAP50']*100 for h in history], 'b-', lw=2, label='mAP@0.50 (%)')
axes[1, 1].plot(epochs_arr, [h['mAP50_95']*100 for h in history], 'orange', lw=2, linestyle='--', label='mAP@0.50:0.95 (%)')
axes[1, 1].set_title('Mean Average Precision (mAP)', fontweight='bold')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

# 6. F1-Score
f1_scores = [(2 * h['precision'] * h['recall']) / max(1e-7, h['precision'] + h['recall']) * 100 for h in history]
axes[1, 2].plot(epochs_arr, f1_scores, 'teal', lw=2, label='F1-Score (%)')
axes[1, 2].set_title('F1-Score Curve', fontweight='bold')
axes[1, 2].set_xlabel('Epoch'); axes[1, 2].legend(); axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
results_img_path = os.path.join(plots_dir, 'results.png')
plt.savefig(results_img_path, dpi=300)
plt.show()
print(f'✅ Grafik results.png tersimpan di: {results_img_path}')

# B. Evaluasi pada Test Set & Confusion Matrix
test_ds = EggYoloRetinaDataset(dataset_root, split='test', imgsz=640, augment=False)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)

model.load_state_dict(torch.load(best_weight_path, map_location=device))
model.eval()

# Inisialisasi Matriks Konfusi 3x3: [0: background/good, 1: cracked, 2: dirt]
cm = np.zeros((3, 3), dtype=np.int32)

with torch.no_grad():
    for imgs, tgts, _ in test_loader:
        preds = model([im.to(device) for im in imgs])
        for p, t in zip(preds, tgts):
            p_b, p_s, p_l = p['boxes'].cpu(), p['scores'].cpu(), p['labels'].cpu()
            g_b, g_l = t['boxes'].cpu(), t['labels'].cpu()
            
            # Filter threshold deteksi 0.30
            mask = p_s >= 0.30
            p_b, p_l = p_b[mask], p_l[mask]
            
            if len(g_b) == 0 and len(p_b) == 0:
                cm[0, 0] += 1  # True negative background
            elif len(g_b) == 0 and len(p_b) > 0:
                for pl in p_l:
                    cm[0, int(pl)] += 1  # False positive dari background
            elif len(g_b) > 0 and len(p_b) == 0:
                for gl in g_l:
                    cm[int(gl), 0] += 1  # False negative (terlewat)
            else:
                ious = box_iou(p_b, g_b)
                matched_g = set()
                for pi, pl in enumerate(p_l):
                    best_iou, best_gi = torch.max(ious[pi], dim=0)
                    if best_iou >= 0.45:
                        matched_g.add(int(best_gi))
                        gl = int(g_l[best_gi])
                        cm[gl, int(pl)] += 1
                    else:
                        cm[0, int(pl)] += 1
                for gi, gl in enumerate(g_l):
                    if gi not in matched_g:
                        cm[int(gl), 0] += 1

# Simpan Confusion Matrix Plot
cm_labels = ['background', 'cracked', 'dirt']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cm_labels, yticklabels=cm_labels, ax=ax1)
ax1.set_title('Confusion Matrix (RetinaNet)', fontweight='bold')
ax1.set_ylabel('True Label'); ax1.set_xlabel('Predicted Label')

# Normalized Confusion Matrix
cm_norm = cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], 1)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=cm_labels, yticklabels=cm_labels, ax=ax2)
ax2.set_title('Normalized Confusion Matrix (RetinaNet)', fontweight='bold')
ax2.set_ylabel('True Label'); ax2.set_xlabel('Predicted Label')

plt.tight_layout()
cm_path = os.path.join(plots_dir, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=300)
plt.show()
print(f'✅ confusion_matrix.png tersimpan di: {cm_path}')

# C. Visualisasi Sampel Prediksi (val_batch_pred.jpg)
sample_imgs, sample_tgts, sample_paths = next(iter(test_loader))
with torch.no_grad():
    sample_preds = model([im.to(device) for im in sample_imgs])

fig, axes = plt.subplots(1, len(sample_imgs), figsize=(16, 4.5))
c_colors = {1: 'red', 2: 'orange'}

for i, (img, pred, pth) in enumerate(zip(sample_imgs, sample_preds, sample_paths)):
    np_img = img.permute(1, 2, 0).cpu().numpy()
    ax = axes[i] if len(sample_imgs) > 1 else axes
    ax.imshow(np_img)
    ax.axis('off')
    for b, s, l in zip(pred['boxes'], pred['scores'], pred['labels']):
        if s >= 0.35:
            x1, y1, x2, y2 = b.cpu().numpy()
            cid = int(l.cpu())
            cname = CLASS_MAP.get(cid, 'defect')
            col = c_colors.get(cid, 'lime')
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=col, lw=2.5)
            ax.add_patch(rect)
            ax.text(x1, max(18, y1 - 6), f'{cname} {s*100:.0f}%', color='white',
                    bbox=dict(facecolor=col, alpha=0.85, pad=1.5), fontsize=9, fontweight='bold')
    ax.set_title(os.path.basename(pth)[:22], fontsize=10)

plt.tight_layout()
val_batch_path = os.path.join(plots_dir, 'val_batch_pred.jpg')
plt.savefig(val_batch_path, dpi=300)
plt.show()
print(f'✅ val_batch_pred.jpg tersimpan di: {val_batch_path}')

In [ ]:
# 7. 📦 Kompres Semua Grafik & Unduh Otomatis 'best.pt' serta Grafik Evaluasi
import zipfile
from google.colab import files

# Buat paket zip berisi seluruh grafik & matriks evaluasi
zip_output_path = '/content/runs/retinanet/retinanet_evaluation_graphs.zip'
with zipfile.ZipFile(zip_output_path, 'w') as zf:
    for fname in ['results.png', 'confusion_matrix.png', 'val_batch_pred.jpg', 'metrics_summary.csv']:
        fpath = os.path.join('/content/runs/retinanet', fname)
        if os.path.exists(fpath):
            zf.write(fpath, arcname=fname)
            print(f'[*] Menambahkan {fname} ke zip...')

print('\n' + '='*75)
print('📥 MENGUNDUH FILE KE KOMPUTER ANDA...')
print('='*75)

# 1. Download file bobot best.pt
if os.path.exists(best_weight_path):
    print('-> Mengunduh weights/best.pt...')
    files.download(best_weight_path)
else:
    print('⚠️ File best.pt tidak ditemukan.')

# 2. Download paket grafik & matriks evaluasi
if os.path.exists(zip_output_path):
    print('-> Mengunduh retinanet_evaluation_graphs.zip...')
    files.download(zip_output_path)
else:
    print('⚠️ File zip grafik tidak ditemukan.')

print('✅ Seluruh proses selesai! File best.pt dan grafik evaluasi siap digunakan untuk komparasi.')